# Conducting customer analytics with Python

In [ ]:
import pandas as pd
import janitor
df = pd.read_csv('../../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv').clean_names(case_type='snake')

In [ ]:
df.shape

In [ ]:
df.head()

# Analytics on Engaged Customers

In [ ]:
df.groupby('response').count()['customer']

In [ ]:
import plotly.express as px

fig = px.bar(
    df.groupby('response').count()['customer'].reset_index(),
    x='response',
    y='customer',
    title='Marketing Engagment',
    color_discrete_sequence=['skyblue'],
    labels={'response': 'Engaged', 'customer': 'Count'}
)

fig.update_layout(showlegend=False, template='ggplot2')
fig.show()

In [ ]:
df.groupby('response').count()['customer']/df.shape[0]

#### Engagement Rates by Offer Type

In [ ]:
by_offer_type_df = df.loc[
    df['response'] == 'Yes'
].groupby([
    'renew_offer_type'
]).count()['customer']/df.groupby('renew_offer_type').count()['customer']

by_offer_type_df

In [ ]:
plot_df = (by_offer_type_df.fillna(0) * 100.0).reset_index(name='Engagement Rate (%)')

fig = px.bar(
    plot_df,
    x='renew_offer_type',
    y='Engagement Rate (%)',
    color_discrete_sequence=['skyblue'],
    template='ggplot2'
)

fig.update_layout(showlegend=False)
fig.show()

#### Offer Type & Vehicle Class

In [ ]:
by_offer_type_df = df.loc[
    df['response'] == 'Yes'
].groupby([
    'renew_offer_type', 'vehicle_class'
]).count()['customer']/df.groupby('renew_offer_type').count()['customer']

by_offer_type_df

In [ ]:
by_offer_type_df = by_offer_type_df.unstack().fillna(0)
by_offer_type_df

In [ ]:
plot_offer_vehicle_df = (
    (by_offer_type_df * 100.0)
    .reset_index()
    .melt(
        id_vars='renew_offer_type',
        var_name='vehicle_class',
        value_name='Engagement Rate (%)'
    )
)

fig = px.bar(
    plot_offer_vehicle_df,
    x='renew_offer_type',
    y='Engagement Rate (%)',
    color='vehicle_class',
    barmode='group',
    template='ggplot2'
)

fig.update_layout(yaxis_title='Engagement Rate (%)')
fig.show()

#### Engagement Rates by Sales Channel

In [ ]:
by_sales_channel_df = df.loc[
    df['response'] == 'Yes'
].groupby([
    'sales_channel'
]).count()['customer']/df.groupby('sales_channel').count()['customer']

by_sales_channel_df

In [ ]:
plot_sales_channel_df = (by_sales_channel_df * 100.0).reset_index(name='Engagement Rate (%)')

fig = px.bar(
    plot_sales_channel_df,
    x='sales_channel',
    y='Engagement Rate (%)',
    color_discrete_sequence=['skyblue'],
    template='ggplot2',
    title='Engagement Rates by Sales Channel'
)

fig.update_layout(showlegend=False)
fig.show()

#### Sales Channel & Vehicle Size

In [ ]:
by_sales_channel_df = df.loc[
    df['response'] == 'Yes'
].groupby([
    'sales_channel', 'vehicle_size'
]).count()['customer']/df.groupby('sales_channel').count()['customer']

by_sales_channel_df

In [ ]:
by_sales_channel_df = by_sales_channel_df.unstack().fillna(0)
by_sales_channel_df

In [ ]:
plot_sales_channel_vehicle_df = (
    (by_sales_channel_df * 100.0)
    .reset_index()
    .melt(
        id_vars='sales_channel',
        var_name='vehicle_size',
        value_name='Engagement Rate (%)'
    )
)

fig = px.bar(
    plot_sales_channel_vehicle_df,
    x='sales_channel',
    y='Engagement Rate (%)',
    color='vehicle_size',
    barmode='group',
    template='ggplot2',
    title='Engagement Rates by Sales Channel and Vehicle Size'
)

fig.show()

#### Engagement Rates by Months Since Policy Inception

In [ ]:
by_months_since_inception_df = df.loc[
    df['response'] == 'Yes'
].groupby(
    by='months_since_policy_inception'
)['response'].count() / df.groupby(
    by='months_since_policy_inception'
)['response'].count() * 100.0

by_months_since_inception_df.fillna(0)

In [ ]:
plot_months_df = by_months_since_inception_df.fillna(0).reset_index(name='Engagement Rate (%)')

fig = px.line(
    plot_months_df,
    x='months_since_policy_inception',
    y='Engagement Rate (%)',
    title='Engagement Rates by Months Since Inception',
    template='ggplot2'
)

fig.update_traces(line=dict(color='skyblue'))
fig.update_layout(
    xaxis_title='Months Since Policy Inception',
    yaxis_title='Engagement Rate (%)'
)

fig.show()

# Customer Segmentation by CLV & Months Since Policy Inception

In [ ]:
df['customer_lifetime_value'].describe()

In [ ]:
df['clv_segment'] = df['customer_lifetime_value'].apply(
    lambda x: 'High' if x > df['customer_lifetime_value'].median() else 'Low'
)

In [ ]:
df['months_since_policy_inception'].describe()

In [ ]:
df['policy_age_segment'] = df['months_since_policy_inception'].apply(
    lambda x: 'High' if x > df['months_since_policy_inception'].median() else 'Low'
)

In [ ]:
df.head()

In [ ]:
plot_segment_df = df.assign(
    segment=df['clv_segment'] + " CLV / " + df['policy_age_segment'] + " Policy Age"
)

fig = px.scatter(
    plot_segment_df,
    x='months_since_policy_inception',
    y='customer_lifetime_value',
    color='segment',
    log_y=True,
    template='ggplot2',  # Plotly template closest to "ggplot"
    color_discrete_map={
        'High CLV / High Policy Age': 'red',
        'Low CLV / High Policy Age': 'blue',
        'High CLV / Low Policy Age': 'orange',
        'Low CLV / Low Policy Age': 'green'
    },
    labels={
        'months_since_policy_inception': 'Months Since Policy Inception',
        'customer_lifetime_value': 'CLV (in log scale)'
    },
    title='Segments by CLV and Policy Age'
)

fig.show()


In [ ]:
engagment_rates_by_segment_df = df.loc[
    df['response'] == 'Yes'
].groupby(
    ['clv_segment', 'policy_age_segment']
).count()['customer']/df.groupby(
    ['clv_segment', 'policy_age_segment']
).count()['customer']

engagment_rates_by_segment_df

In [ ]:
plot_segment_engagement_df = (
    (engagment_rates_by_segment_df * 100.0)
    .reset_index(name='Engagement Rate (%)')
)

fig = px.bar(
    plot_segment_engagement_df,
    x='clv_segment',
    y='Engagement Rate (%)',
    color='policy_age_segment',
    barmode='group',
    template='ggplot2',
    title='Engagement Rates by Customer Segments',
    labels={
        'clv_segment': 'CLV Segment',
        'policy_age_segment': 'Policy Age Segment'
    },
    width=800,
    height=500
)

fig.show()